# TD-MW-RPLS: Time-Difference Moving-Window Recursive PLS
Based on: Fu et al. (2017). *Measurement Science and Technology*, 28(4).

## 0. Cài đặt thư viện (nếu cần)

In [ ]:
# !pip install numpy pandas scikit-learn matplotlib

## 1. Import thư viện

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

## 2. Load dữ liệu của bạn

**Định dạng CSV yêu cầu:**
- Cột đầu tiên: `time` (hoặc index thời gian)
- Các cột giữa: biến đầu vào `x1, x2, ..., xm`
- Cột cuối cùng: biến mục tiêu `y`

> Nếu file của bạn dùng delimiter khác (`,` hoặc `;`), chỉnh tham số `sep` bên dưới.

In [ ]:
# ─── THAY ĐỔI CÁC THAM SỐ NÀY ───────────────────────────────────────────────

CSV_PATH  = 'your_file.csv'   # << đường dẫn tới file CSV của bạn
SEP       = ';'               # dấu phân cách: ',' hoặc ';'
DECIMAL   = ','               # dấu thập phân: '.' hoặc ','
Y_COLUMN  = None              # tên cột target, None = lấy cột cuối tự động
TIME_COL  = None              # tên cột thời gian, None = bỏ qua
DROP_COLS = []                # danh sách cột cần loại bỏ, ví dụ: ['col1', 'col2']

# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(CSV_PATH, sep=SEP, decimal=DECIMAL)
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# Xử lý cột thời gian và cột không cần thiết
if TIME_COL and TIME_COL in df.columns:
    df = df.drop(columns=[TIME_COL])

for c in DROP_COLS:
    if c in df.columns:
        df = df.drop(columns=[c])

# Xác định X và y
cols = list(df.columns)
if Y_COLUMN and Y_COLUMN in cols:
    y_col  = Y_COLUMN
    x_cols = [c for c in cols if c != y_col]
else:
    y_col  = cols[-1]
    x_cols = cols[:-1]

# Loại bỏ hàng có giá trị NaN
df = df.dropna()

X = df[x_cols].values.astype(float)
y = df[y_col].values.astype(float)

print(f'Features ({len(x_cols)}): {x_cols}')
print(f'Target: {y_col}')
print(f'Shape: X={X.shape}, y={y.shape}')

## 3. Cấu hình tham số mô hình

In [ ]:
# ─── ĐIỀU CHỈNH THAM SỐ MÔ HÌNH ─────────────────────────────────────────────

WINDOW     = 50   # kích thước cửa sổ trượt
LAG        = 1    # bước sai phân thời gian (thường dùng 1)
N_COMP     = 2    # số thành phần PLS (latent variables)

# ─────────────────────────────────────────────────────────────────────────────

print(f'Config: window={WINDOW}, lag={LAG}, n_components={N_COMP}')
print(f'Minimum samples needed: {WINDOW + LAG}')
print(f'Available samples: {len(y)}')
assert len(y) >= WINDOW + LAG + 10, \
    f'Cần ít nhất {WINDOW + LAG + 10} mẫu, hiện có {len(y)}. Giảm WINDOW hoặc LAG.'

## 4. Định nghĩa các lớp mô hình

In [ ]:
def _scale(arr_2d):
    mu = arr_2d.mean(axis=0)
    sd = np.maximum(arr_2d.std(axis=0), 1e-8)
    return (arr_2d - mu) / sd, mu, sd

def _scale1d(vec):
    mu = vec.mean()
    sd = max(float(vec.std()), 1e-8)
    return (vec - mu) / sd, mu, sd


class MW_RPLS:
    """Moving-Window Recursive PLS (baseline, không dùng time-diff)."""

    def __init__(self, window_size=20, n_components=3):
        self.window_size  = window_size
        self.n_components = n_components
        self.model_       = None
        self.window_X_    = []
        self.window_y_    = []
        self.n_updates_   = 0

    def _fit(self):
        X = np.array(self.window_X_)
        y = np.array(self.window_y_).reshape(-1, 1)
        Xs, _, _ = _scale(X)
        ys, _, _ = _scale(y)
        nc = max(1, min(self.n_components, Xs.shape[1], Xs.shape[0] - 1))
        pls = PLSRegression(n_components=nc, max_iter=500)
        pls.fit(Xs, ys)
        return pls

    def _predict_one(self, x_t):
        X = np.array(self.window_X_)
        y = np.array(self.window_y_)
        mu_X, sd_X = X.mean(0), np.maximum(X.std(0), 1e-8)
        mu_y, sd_y = float(y.mean()), max(float(y.std()), 1e-8)
        xs = (x_t - mu_X) / sd_X
        ys_hat = self.model_.predict(xs.reshape(1, -1))[0, 0]
        return float(ys_hat * sd_y + mu_y)

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y, float).ravel()
        self.window_X_ = list(X[:self.window_size])
        self.window_y_ = list(y[:self.window_size])
        self.model_    = self._fit()
        self.n_updates_ = 1
        return self

    def train_online(self, X, y, verbose=True):
        X, y = np.asarray(X, float), np.asarray(y, float).ravel()
        n    = len(X)
        y_pred_all   = np.full(n, np.nan)
        update_flags = np.zeros(n, bool)

        for t in range(self.window_size, n):
            y_pred_all[t] = self._predict_one(X[t])
            self.window_X_.append(X[t])
            self.window_y_.append(y[t])
            if len(self.window_X_) > self.window_size:
                self.window_X_.pop(0)
                self.window_y_.pop(0)
            self.model_ = self._fit()
            self.n_updates_ += 1
            update_flags[t] = True

        if verbose:
            valid = ~np.isnan(y_pred_all)
            rmse = np.sqrt(mean_squared_error(y[valid], y_pred_all[valid]))
            r2   = r2_score(y[valid], y_pred_all[valid])
            print(f'[MW-RPLS]    Samples: {n - self.window_size} | Updates: {self.n_updates_} | RMSE: {rmse:.6f} | R²: {r2:.6f}')

        return dict(y_pred=y_pred_all, y_true=y, update_flags=update_flags)


class TD_MW_RPLS:
    """Time-Difference Moving-Window Recursive PLS với Adaptive Model Updating."""

    def __init__(self, window_size=20, n_components=4,
                 time_diff_lag=1, adaptive_update=True):
        self.window_size  = window_size
        self.n_components = n_components
        self.lag          = time_diff_lag
        self.adaptive     = adaptive_update
        self.model_       = None
        self.window_dX_   = []
        self.window_dy_   = []
        self.confidence_  = None
        self.n_updates_   = 0

    @staticmethod
    def _diff(arr, lag):
        return arr[lag:] - arr[:-lag]

    def _window_arrays(self):
        return np.array(self.window_dX_), np.array(self.window_dy_)

    def _fit_pls(self):
        dX, dy = self._window_arrays()
        dX_s, _, _  = _scale(dX)
        dy_s, _, _  = _scale1d(dy)
        nc = max(1, min(self.n_components, dX_s.shape[1], dX_s.shape[0] - 1))
        pls = PLSRegression(n_components=nc, max_iter=500)
        pls.fit(dX_s, dy_s)
        return pls

    def _compute_confidence(self):
        dX, dy = self._window_arrays()
        dX_s, _, _  = _scale(dX)
        dy_s, _, _  = _scale1d(dy)
        res   = dy_s - self.model_.predict(dX_s).ravel()
        denom = max(len(res) - self.model_.n_components - 1, 1)
        return float(np.sqrt((res ** 2).sum() / denom))

    def _predict_one(self, x_t, x_prev, y_prev):
        dX, dy = self._window_arrays()
        mu_X, sd_X = dX.mean(0), np.maximum(dX.std(0), 1e-8)
        mu_y, sd_y = float(dy.mean()), max(float(dy.std()), 1e-8)
        dx_s     = (x_t - x_prev - mu_X) / sd_X
        dy_s_hat = float(self.model_.predict(dx_s.reshape(1, -1)).ravel()[0])
        delta_y  = dy_s_hat * sd_y + mu_y
        return float(y_prev + delta_y)

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y, float).ravel()
        need = self.window_size + self.lag
        if len(X) < need:
            raise ValueError(f'Cần >= {need} mẫu, có {len(X)}')
        dX = self._diff(X[:need], self.lag)
        dy = self._diff(y[:need], self.lag)
        self.window_dX_ = list(dX)
        self.window_dy_ = list(dy)
        self.model_     = self._fit_pls()
        self.n_updates_ = 1
        self.confidence_= self._compute_confidence()
        return self

    def train_online(self, X, y, verbose=True):
        X, y = np.asarray(X, float), np.asarray(y, float).ravel()
        n    = len(X)
        start = self.window_size + self.lag

        y_pred_all   = np.full(n, np.nan)
        update_flags = np.zeros(n, bool)
        conf_history = np.full(n, np.nan)
        errors       = np.full(n, np.nan)

        for t in range(start, n):
            x_t,    y_t    = X[t],          y[t]
            x_prev, y_prev = X[t - self.lag], y[t - self.lag]
            y_hat         = self._predict_one(x_t, x_prev, y_prev)
            y_pred_all[t] = y_hat
            err           = abs(y_t - y_hat)
            errors[t]     = err

            self.window_dX_.append(x_t   - x_prev)
            self.window_dy_.append(y_t   - y_prev)
            if len(self.window_dX_) > self.window_size:
                self.window_dX_.pop(0)
                self.window_dy_.pop(0)

            if (not self.adaptive) or (err > self.confidence_):
                self.model_      = self._fit_pls()
                self.confidence_ = self._compute_confidence()
                self.n_updates_ += 1
                update_flags[t]  = True

            conf_history[t] = self.confidence_

        if verbose:
            valid = ~np.isnan(y_pred_all)
            rmse  = np.sqrt(mean_squared_error(y[valid], y_pred_all[valid]))
            r2    = r2_score(y[valid], y_pred_all[valid])
            print(f'[TD-MW-RPLS] Samples: {n - start} | Updates: {self.n_updates_} | δₑ: {self.confidence_:.6f} | RMSE: {rmse:.6f} | R²: {r2:.6f}')

        return dict(y_pred=y_pred_all, y_true=y,
                    update_flags=update_flags,
                    confidence_history=conf_history,
                    errors=errors)

print('Classes defined ✅')

## 5. Huấn luyện mô hình TD-MW-RPLS (Adaptive)

In [ ]:
model = TD_MW_RPLS(
    window_size    = WINDOW,
    n_components   = N_COMP,
    time_diff_lag  = LAG,
    adaptive_update= True
)

model.fit(X, y)
print(f'Initial δₑ = {model.confidence_:.6f}')

results = model.train_online(X, y, verbose=True)

## 6. Vẽ đồ thị kết quả

In [ ]:
def plot_results(results, title_suffix='', save_path=None):
    y_true, y_pred = results['y_true'], results['y_pred']
    conf    = results.get('confidence_history')
    updates = results['update_flags']

    valid   = ~np.isnan(y_pred)
    idx     = np.where(valid)[0]
    rmse    = np.sqrt(mean_squared_error(y_true[valid], y_pred[valid]))
    r2      = r2_score(y_true[valid], y_pred[valid])
    rel_err = (y_pred[valid] - y_true[valid]) / (np.abs(y_true[valid]) + 1e-8)

    has_conf = conf is not None and not np.all(np.isnan(conf[valid]))
    nrows    = 3 if has_conf else 2
    heights  = [3, 2, 1.5] if has_conf else [3, 2]

    fig, axes = plt.subplots(nrows, 1, figsize=(14, 4 * nrows),
                             gridspec_kw={'height_ratios': heights})
    fig.suptitle(f'TD-MW-RPLS  {title_suffix}\nRMSE = {rmse:.4f}   R² = {r2:.4f}',
                 fontsize=13, fontweight='bold')

    ax = axes[0]
    ax.plot(idx, y_true[valid], color='#2c7bb6', lw=1.5, label='Real value')
    ax.plot(idx, y_pred[valid], color='#d7191c', lw=1.2, ls='--',
            alpha=0.85, label='Predicted value')
    ylim = ax.get_ylim()
    ax.vlines(np.where(updates)[0], *ylim, color='gray', alpha=0.12, lw=0.7,
              label='Model update')
    ax.set_ylim(ylim)
    ax.set_ylabel('Output y')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(alpha=0.3)

    ax2 = axes[1]
    ax2.plot(idx, rel_err, color='#fd8d3c', lw=1.0)
    ax2.axhline(0, color='black', lw=0.8)
    ax2.fill_between(idx, rel_err, 0, alpha=0.25, color='#fd8d3c')
    ax2.set_ylabel('Relative error')
    ax2.grid(alpha=0.3)

    if has_conf:
        ax3 = axes[2]
        ax3.plot(idx, conf[valid], color='#31a354', lw=1.2, label='δₑ')
        ax3.set_ylabel('Confidence limit δₑ')
        ax3.set_xlabel('Observation number')
        ax3.legend(fontsize=9)
        ax3.grid(alpha=0.3)
    else:
        axes[-1].set_xlabel('Observation number')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Plot saved → {save_path}')
    plt.show()


plot_results(results, title_suffix='(Adaptive, your data)')

## 7. So sánh 4 mô hình

In [ ]:
def compare_models(X, y, window_size=20, n_components=4, lag=1):
    start = window_size + lag
    rows  = []

    for name, model_type in [
        ('PLS (static)',           'static'),
        ('MW-RPLS',                'mw'),
        ('TD-MW-RPLS',             'td'),
        ('TD-MW-RPLS + Adaptive',  'td_adaptive'),
    ]:
        if model_type == 'static':
            dX_fit = X[lag:window_size + lag] - X[:window_size]
            dy_fit = y[lag:window_size + lag] - y[:window_size]
            dX_s, mu_X, sd_X = _scale(dX_fit)
            dy_s, mu_y, sd_y = _scale1d(dy_fit)
            nc  = max(1, min(n_components, dX_s.shape[1], dX_s.shape[0] - 1))
            pls = PLSRegression(n_components=nc, max_iter=500).fit(dX_s, dy_s.reshape(-1,1))
            preds = []
            for t in range(start, len(X)):
                dx_s   = (X[t] - X[t - lag] - mu_X) / sd_X
                dy_hat = float(pls.predict(dx_s.reshape(1, -1))[0, 0])
                preds.append(y[t - lag] + dy_hat * sd_y + mu_y)
            y_true = y[start:]
            y_pred = np.array(preds)
            n_upd  = 0

        elif model_type == 'mw':
            m = MW_RPLS(window_size=window_size, n_components=n_components)
            m.fit(X, y)
            res   = m.train_online(X, y, verbose=False)
            valid = ~np.isnan(res['y_pred'])
            mask  = valid & (np.arange(len(y)) >= start)
            y_true = res['y_true'][mask]
            y_pred = res['y_pred'][mask]
            n_upd  = m.n_updates_

        else:
            adaptive = (model_type == 'td_adaptive')
            m = TD_MW_RPLS(window_size=window_size, n_components=n_components,
                           time_diff_lag=lag, adaptive_update=adaptive)
            m.fit(X, y)
            res   = m.train_online(X, y, verbose=False)
            valid = ~np.isnan(res['y_pred'])
            y_true = res['y_true'][valid]
            y_pred = res['y_pred'][valid]
            n_upd  = m.n_updates_

        rmse    = np.sqrt(mean_squared_error(y_true, y_pred))
        r2      = r2_score(y_true, y_pred)
        rel_err = (y_pred - y_true) / (np.abs(y_true) + 1e-8)

        rows.append({
            'Model'           : name,
            'Max |rel error|' : round(float(np.max(np.abs(rel_err))), 4),
            'Min rel error'   : round(float(np.min(rel_err)), 4),
            'RMSE'            : round(rmse, 4),
            'R²'              : round(r2, 4),
            'Model updates'   : n_upd,
        })
        print(f'  {name:40s}  RMSE={rmse:.4f}  R²={r2:.4f}  updates={n_upd}')

    return pd.DataFrame(rows)


print('Comparing models ...')
comp_df = compare_models(X, y, window_size=WINDOW, n_components=N_COMP, lag=LAG)
print()
comp_df

In [ ]:
# Visualise comparison as bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

comp_df.plot.bar(x='Model', y='RMSE', ax=axes[0], legend=False,
                 color='#2c7bb6', edgecolor='white')
axes[0].set_title('RMSE (lower is better)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)
axes[0].grid(axis='y', alpha=0.4)

comp_df.plot.bar(x='Model', y='R²', ax=axes[1], legend=False,
                 color='#31a354', edgecolor='white')
axes[1].set_title('R² (higher is better)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.show()

## 8. Lưu kết quả

In [ ]:
# Lưu bảng so sánh
comp_df.to_csv('model_comparison.csv', index=False)

# Lưu predictions
valid = ~np.isnan(results['y_pred'])
pred_df = pd.DataFrame({
    'y_true'     : results['y_true'][valid],
    'y_pred'     : results['y_pred'][valid],
    'error'      : results['errors'][valid],
    'model_update': results['update_flags'][valid],
    'delta_e'    : results['confidence_history'][valid],
})
pred_df.to_csv('predictions.csv', index=False)

# Lưu plot
plot_results(results, title_suffix='(Adaptive, your data)', save_path='td_mw_rpls_results.png')

print('Saved: model_comparison.csv, predictions.csv, td_mw_rpls_results.png ✅')